In [1]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# Lab 06: Exact Inference

In this session, we explore the sum-product and the variable elimination algorithms for exact inference. The goal of the first part is to fully understand the intuition behind the sum-product algorithm. We propose implementations for both algorithms.

_Note: As usual, questions marked with a (*) can be presented in class in exchange for up to 1 bonus point._

## Part 1: The Sum-Product algorithm

In this part, we consider an undirected tree. In the following question, we show that the case of directed trees also falls into this assumption. 

#### Question 1 (*)
Let $p(\mathbf{x})$ a distribution which factorizes over a directed tree. Show that there exists an undirected tree over which $p(\mathbf{x})$ factorizes.

### <font color='green'><u>Solution</u></font>
Since $p(\mathbf x)$ factorizes over a directed tree $G = (V,E)$, it can be written as $p(\mathbf x) = p(r) \prod_{(i,j) \in E} p(x_j \mid x_i)$, where $r$ is the root of the tree. The symmetrized graph $\widetilde G$ is an undirected tree and a factorization has to be of the form $\frac 1 Z \prod_{C\in \mathcal C} \psi_C(x_C)$, where $\mathcal C$ is the set of cliques. Since $\widetilde G$ is an undirected tree, we have $\mathcal C = \widetilde E \cup \{\{v\} \mid v \in V\}$. So by setting
$$
\psi_{ij}(x_i,x_j) = \begin{cases}
p(x_j \mid x_i) &\text{if } (i,j) \in E, \\
p(x_i \mid x_j) &\text{if } (j,i) \in E,
\end{cases}
$$
and 
$$\psi_i(x_i) = \begin{cases}
p(x_i) &\text{if } x_i = r,\\
1 &\text{else},
\end{cases}$$
as well as $Z = 1$, we see that $p$ factorizes over $\widetilde G$.
<hr>


### Understanding the algorithm

We remind that the sum-product algorithm is based on the idea of passing messages from one node to the other, in order to compute some marginal distribution $p(x_i)$. The order of the nodes is determined by the structure of the tree, where node $X_i$ is taken as the root. 

We remind that the messages are defined by:
$$
\mu_{j \rightarrow i}(x_i) = \sum_{x_j} \psi_{ij}(x_i, x_j) \psi_j(x_j) \prod_{k \in \mathcal{C}(j)} \mu_{k \rightarrow j}(x_j)
$$
where $\mathcal{C}(j)$ are the children of node $X_j$ and with the convention that $\prod_{k \in \emptyset} = 1$.

#### Question 2 (*)

Consider a chain of size 3, with edges only $(1,2)$ and $(2,3)$.

a) Show that $p(x_1) = \frac{1}{Z} \psi_1(x_1) \mu_{2 \rightarrow 1}(x_1)$.<br>
b) For this same chain, check that $p(x_2) = \frac{1}{Z} \psi_2(x_2) \mu_{3 \rightarrow 2}(x_2) \mu_{1 \rightarrow 2}(x_2)$.

### <font color='green'><u>Solution</u></font>
a) First, write $p(x_1)$ by first marginalizing, then factorizing:
\begin{align*}
    p(x_1) &= \sum_{x_2}\sum_{x_3} p(\mathbf x) = \sum_{x_2} \sum_{x_3} \frac 1 Z \psi_1(x_1) \psi_2(x_2) \psi_3(x_3) \psi_{12}(x_1,x_2) \psi_{23}(x_2,x_3) \\
    &= \frac 1 Z \psi_1(x_1) \sum_{x_2} \psi_2(x_2) \psi_{12}(x_1,x_2) \sum_{x_3} \psi_3(x_3) \psi_{23}(x_2,x_3).
\end{align*}
Next, let's write out $\mu_{2 \rightarrow 1}$ using the definition above:
\begin{align*}
\mu_{2 \rightarrow 1}(x_1) = \sum_{x_2} \psi_{12}(x_1,x_2) \psi_2(x_2) \textcolor{red}{\mu_{3 \rightarrow 2}(x_2)}
= \sum_{x_2} \psi_{12}(x_1,x_2) \psi_2(x_2) \textcolor{red}{\sum_{x_3} \psi_{23}(x_2,x_3) \psi_3(x_3)}.
\end{align*}
We see that $\frac 1 Z \psi_1(x_1) \mu_{2 \rightarrow 1}(x_1)$ is the same as the first expression.

b) We proceed similar to above. First, the naive computation:
\begin{align*}
    p(x_2) &= \sum_{x_1}\sum_{x_3} p(\mathbf x) = \sum_{x_1} \sum_{x_3} \frac 1 Z \psi_1(x_1) \psi_2(x_2) \psi_3(x_3) \psi_{12}(x_1,x_2) \psi_{23}(x_2,x_3) \\
    &= \frac 1 Z \psi_2(x_2) \sum_{x_1} \psi_1(x_1) \psi_{12}(x_1,x_2) \sum_{x_3} \psi_3(x_3) \psi_{23}(x_2,x_3).
\end{align*}

Next, we re-use from above that $\mu_{3\rightarrow 2}(x_2,x_3) = \sum_{x_3} \psi_{23}(x_2,x_3) \psi_2(x_2)$. Additionally, we have $\mu_{1 \rightarrow 2} = \sum_{x_1} \psi_{12}(x_1,x_2) \psi_1(x_1)$. Notice that $2$ is considered a parent of $1$ in the second expression, so that the product over $1$'s children ends up being empty.
Since these are the two factors of the result of the naive marginalization apart from $\frac 1 Z \psi_2(x_2)$, we conclude that $p(x_2) = \frac 1 Z \psi_{2}(x_2) \mu_{3\rightarrow 2}(x_2) \mu_{1\rightarrow 2}(x_2)$.

<hr>

The sum-product algorithm works by computing messages from the leaves to the root. The final messages are used to compute:
$$
p(x_i) = \frac{1}{Z} \psi_i(x_i) \prod_{j \in \mathcal{C}(i)} \mu_{j \rightarrow i}(x_i)
$$


### Implementation of the algorithm

We first need to represent the undirected tree. For this, we separately represent the edges, in the form of a binary matrix, and the unary and binary potentials. 

Unary potentials are stored in a dictionary of a arrays, such that ```psi_u[i][k]```represents $\psi_i(k)$ (assuming that $X_i$ takes values in $\lbrace 1, \dotsc, K_i \rbrace$.  

Unary potentials are stored in a dictionary of arrays, such that ```psi_b[i,j][k,l]```represents $\psi_{ij}(k,l)$.

**Remark:** Note that in the code we define $\psi_{ij}$ and $\psi_{ji}$ while, mathematically speaking, the order of the nodes in the clique does not matter (mathematically, we have $\psi_{ij}(x_i, x_j) = \psi_{ji}(x_i, x_j)$). We make this choice because we want that ```psi_b[i,j][k,l]``` corresponds to the evaluation of $\psi_{ij}$ for $x_i = k$ and $x_j = l$, and ```psi_b[j,i][k,l]``` corresponds to the evaluation of $\psi_{ij}$ for $x_j = k$ and $x_i = l$. This implies that ```psi_b[j,i]``` is the transpose of ```psi_b[i,j]```.

In [3]:
edges = np.array([ [0, 1, 0, 0, 0], [1, 0, 1, 1, 0], [0, 1, 0, 0, 0], [0, 1, 0, 0, 1], [0, 0, 0, 1, 0] ])

psi_u = dict()
psi_u[0] = np.array([.6, .4])
psi_u[1] = np.array([1., 1.])
psi_u[2] = np.array([1., 1.])
psi_u[3] = np.array([1., 1.])
psi_u[4] = np.array([1., 1.])


psi_b = dict()

# p(x_1 = 0 | x_0 = 0) = 0.9
# p(x_1 = 0 | x_0 = 1) = 0.2

psi_b[0, 1] = np.array([[.9, .1], [.2, .8]])
psi_b[1, 0] = np.array([[.9, .2], [.1, .8]])

# p(x_2 = 0 | x_1 = 0) = 0.6
# p(x_2 = 0 | x_1 = 1) = 0.5

psi_b[2, 1] = np.array([[.6, .5], [.4, .5]])
psi_b[1, 2] = np.array([[.6, .4], [.5, .5]])


# p(x_3 = 0 | x_1 = 0) = 0.3
# p(x_3 = 0 | x_1 = 1) = 0.6

psi_b[3, 1] = np.array([[.3, .6], [.7, .4]])
psi_b[1, 3] = np.array([[.3, .7], [.6, .4]])

# p(x_4 = 0 | x_3 = 0) = 0.75
# p(x_4 = 0 | x_3 = 1) = 0.3

psi_b[4, 3] = np.array([[.75, .3], [.25, .7]])
psi_b[3, 4] = np.array([[.75, .25], [.3, .7]])


print(psi_u)
print(psi_b)

{0: array([0.6, 0.4]), 1: array([1., 1.]), 2: array([1., 1.]), 3: array([1., 1.]), 4: array([1., 1.])}
{(0, 1): array([[0.9, 0.1],
       [0.2, 0.8]]), (1, 0): array([[0.9, 0.2],
       [0.1, 0.8]]), (2, 1): array([[0.6, 0.5],
       [0.4, 0.5]]), (1, 2): array([[0.6, 0.4],
       [0.5, 0.5]]), (3, 1): array([[0.3, 0.6],
       [0.7, 0.4]]), (1, 3): array([[0.3, 0.7],
       [0.6, 0.4]]), (4, 3): array([[0.75, 0.3 ],
       [0.25, 0.7 ]]), (3, 4): array([[0.75, 0.25],
       [0.3 , 0.7 ]])}


Note that we don't define the potentials for edges that don't exist. 


#### Question 3 (*)

Manually compute $p(x_2)$ for this defined network.

### <font color='green'><u>Solution</u></font>
First, let's marginalize and rearrange the terms:

$$p(x_2) = \sum_{x_1} \sum_{x_0} \sum_{x_3} \sum_{x_4} p(\mathbf x) = \frac 1 Z \psi_2(x_2) \underbrace{\sum_{x_1} \psi_1(x_1) \psi_{12}(x_1,x_2) \underbrace{\sum_{x_0} \psi_0(x_0) \psi_{01}(x_0,x_1)}_{\mu_{0 \to 1}(x_1)} \underbrace{\sum_{x_3} \psi_3(x_3) \psi_{13}(x_1,x_3) \underbrace{\sum_{x_4} \psi_4(x_4) \psi_{34}(x_3,x_4)}_{\mu_{4 \to 3}(x_3)}}_{\mu_{3 \to 1}(x_1)}}_{\mu_{1 \to 2}(x_2)}$$

For clarity, we have marked the messages with braces, showcasing how the messages are derived from the factoring over the tree.

Finally, we can compute this from right to left: 

In [4]:
mu_4_3 = psi_b[3,4] @ psi_u[4]
mu_3_1 = psi_b[1,3] @ (mu_4_3 * psi_u[3])
mu_0_1 = psi_b[1,0] @ psi_u[0]
mu_1_2 = psi_b[2,1] @ (mu_0_1 * mu_3_1 * psi_u[1])

res = mu_1_2 * psi_u[2]
res /= res.sum()

print(res)  # first component: p(x_2 = 0), second component: p(x_2 = 1)

[0.562 0.438]


We will now implement the Sum-Product algorithm step by step. The first step of the algorithm is to compute the messages recursively. 

#### Question 4 (*)

a) Implement a method computing the message. For this, we can use the (already implemented) method ```get_children(node, parent, edges)``` which finds the children of a node, knowing its parent (note that the parent is unique, since we are in a tree).<br>
b) Implement the sum-product algorithm, computing the probability $p(x_i)$ for a given node $i$.

### <font color='green'><u>Solution</u></font>

In [5]:
def get_children(node: int, parent: int, edges: np.ndarray) -> set[int]:
    neighbors = set(np.nonzero(edges[node,:])[0])
    if parent in neighbors: neighbors.remove(parent)
    return neighbors

def message(j: int, i: int, edges: np.ndarray) -> np.ndarray:
    N_i = len(psi_u[i])
    mu = np.zeros((N_i,))
    ### SOLUTION
    children = get_children(j, i, edges)
    
    N_j = len(psi_u[j])
    prod = np.ones((N_j,))
    for k in children:
        prod *= message(k, j, edges)

    mu = (psi_b[i,j] * psi_u[j] * prod).sum(axis=1)
    return mu

In [6]:
def sum_product(i: int, edges: np.ndarray) -> np.ndarray:
    proba = np.copy(psi_u[i])
    for j in get_children(i, None, edges):
        proba *= message(j, i, edges)
    return proba / np.sum(proba)

In [7]:
sum_product(2, edges)

array([0.562, 0.438])

## Part 2: Variable elimination

We will now propose an implementation of the variable elimination algorithm. 

Unlike for the sum-product algorithm, we now need to track the potentials, and consequently we will not duplicate the potentials $\psi_{ij}$ and $\psi_{ji}$. We will take the convention that we take the potentials with the variables ordered from the smallest to the larget. The function to order the indices is ```order_indices(I)```. 

In [8]:
def order_indices(I):
    return tuple(sorted(I))

We start by instantiating the graph. 

In [9]:
edges = np.array([ [0, 1, 1, 0, 0], [1, 0, 0, 1, 0], [1, 0, 0, 1, 0], [0, 1, 1, 0, 1], [0, 0, 0, 1, 0] ])

psi = dict()
psi[0, 1] = psi_b[0, 1] = np.array([[.9, .1], [.2, .8]])
psi[1, 3] = np.array([[.3, .7], [.6, .4]])
psi[0, 2] = np.array([[.6, .4], [.5, .5]])
psi[2, 3] = np.array([[.75, .3], [.25, .7]])
psi[3, 4] = np.array([[.9, .05], [1.2, .3]])

In the initialization phase, we first store all the potentials into the list of active potentials.

In [11]:
def add_to_active_potentials(active_potentials, indices, potential):
    if indices in active_potentials:
        active_potentials[indices].append(potential)
    else:
        active_potentials[indices] = [potential]


def remove_from_active_potentials(active_potentials, indices):
    if indices in active_potentials:
        active_potentials.pop(indices)

        
def initialize(psi):
    active_potentials = dict()
    for indices in psi:
        add_to_active_potentials(active_potentials, indices, psi[indices])
    return active_potentials

active_potentials = initialize(psi)
print(active_potentials)

{(0, 1): [array([[0.9, 0.1],
       [0.2, 0.8]])], (1, 3): [array([[0.3, 0.7],
       [0.6, 0.4]])], (0, 2): [array([[0.6, 0.4],
       [0.5, 0.5]])], (2, 3): [array([[0.75, 0.3 ],
       [0.25, 0.7 ]])], (3, 4): [array([[0.9 , 0.05],
       [1.2 , 0.3 ]])]}


We then store the evidence as part of the active potentials. 

In [12]:
evidence = dict({1: 1,
                 4: 0})

def collect_evidence(evidence, active_potentials):
    for v in evidence:
        value = evidence[v]
        add_to_active_potentials(active_potentials, (v,), np.array([1-value, value]))

collect_evidence(evidence, active_potentials)
active_potentials

{(0,
  1): [array([[0.9, 0.1],
         [0.2, 0.8]])],
 (1,
  3): [array([[0.3, 0.7],
         [0.6, 0.4]])],
 (0,
  2): [array([[0.6, 0.4],
         [0.5, 0.5]])],
 (2,
  3): [array([[0.75, 0.3 ],
         [0.25, 0.7 ]])],
 (3,
  4): [array([[0.9 , 0.05],
         [1.2 , 0.3 ]])],
 (1,): [array([0, 1])],
 (4,): [array([1, 0])]}

We now need to implement the elimination operation itself (called update step, in the lecture). For this we will need to define the new potentials over the set of all referenced potentials. 

#### Question 5 (*)

Implement the update step. In the definition of the method, $n$ is the index of the variable to eliminate.

### <font color='green'><u>Solution</u></font>

In [14]:
def update(n, active_potentials):
    
    updated_potentials = deepcopy(active_potentials)
    
    # Find all potentials involving n
    
    potentials_of_interest = []
    involved_variables = set()
    
    for indices in active_potentials:
        if n in indices:
            involved_variables.update(indices) # Add the variables to the set of involved variables
            potentials_of_interest.append((indices, active_potentials[indices]))
            updated_potentials.pop(indices)
    
    involved_variables = order_indices(involved_variables)
    
    # Compute phi:
    
    phi_shape = tuple((2 for i in involved_variables))
    phi = np.ones(phi_shape)
    
    ### SOLUTION

    for indices, potential in potentials_of_interest:
        if n not in indices:
            continue
        for pot in potential:
            slice_tup = tuple(slice(None) if i in indices else None for i in involved_variables)
            tmp = pot[slice_tup]
            phi *= tmp
    
    # Compute tau_n:
    
    involved_variables_minus_n = tuple(i for i in involved_variables if i != n)
    idx = sum([i if j == n else 0 for i,j in enumerate(involved_variables)])
    tau_n = phi.sum(axis=idx)

    add_to_active_potentials(updated_potentials, involved_variables_minus_n, tau_n)
    return updated_potentials

#### Question 6 (*)

Use the update function to compute the probability $p(x_F | \bar{x}_E)$.

### <font color='green'><u>Solution</u></font>

In [15]:
def variable_elimination(F, evidence, psi, ordering):
    # 1. Initialize
    active_potentials = initialize(psi)

    # 2. Evidence
    collect_evidence(evidence, active_potentials)

    # 3. Update
    for n in ordering:
        if n in F:
            continue
        active_potentials = update(n, active_potentials)

    # 3.5. Multiply the remaining factors (which, by construction, only involve variables in F)
    potential = np.ones(tuple(2 for i in F))
    for indices,pot in active_potentials.items():
        for p in pot:
            potential *= p[tuple(slice(None) if i in indices else None for i in F)]

    # 4. Normalize
    return potential / potential.sum()

In [16]:
variable_elimination((0, 3), evidence, psi, [4, 2, 1, 0, 3])

(2, 2) 2
(2, 2, 2) 3
(2, 2, 2) 3


array([[0.06459611, 0.04802297],
       [0.4697899 , 0.41759102]])